# 5 · The probe region and GITT

The experimental cell has an open-electrolyte region between the Zn anode and the separator where a pH probe sits. Here it is a mesh region of porosity 1 and length `L_probe` (0 removes it). It also holds most of the electrolyte, so it acts as a reservoir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {f"L_probe = {L} cm": run(Params(L_probe=L, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for L in (0.0, 0.1, 0.5)}
summary(runs)
compare(runs, "probe region length", cols=("V", "pH_cath", "Mn_cath_M"));

A longer probe region dilutes the Mn²⁺ released on discharge, and less of it comes back to the cathode for deposition on charge.

## GITT: what a probe between anode and separator sees

Pulses of 720 s at 50 mA/g and rests of 1800 s. The probe reads the electrolyte near the anode, not the cathode's.

In [ ]:
r = run(Params(steps="cc I=50 t=720; rest t=1800", cycles=1000, write_interval=30.0))
t = r.column("t_h")
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
ax[0].plot(t, r.column("V"), lw=0.8); ax[0].set(ylabel="V")
for c in ("pH_cath", "pH_probe", "pH_anode"):
    ax[1].plot(t, r.column(c), lw=0.8, label=c)
ax[1].axhline(5.41, ls=":", c="gray", label="ZHS saturation (2 M ZnSO4)")
ax[1].set(xlabel="time [h]", ylabel="pH"); ax[1].legend(fontsize=8);

Before ZHS forms, the cathode pH rises during each pulse and relaxes during each rest; once ZHS is present the cathode pH stays near saturation through pulses and rests, as Bernard et al. (2025) inferred from the rest voltages. The probe follows more slowly.